# 10 · Convex hull, checked against shapely

## What this checks

**Convex hull** draws the smallest convex shape that holds every point. It is like a rubber band stretched around
them. It also reports the shape's area and its span, the longest distance across it. This check covers release
0.4.0.

Fieldwork says how it does this. It projects the points to UTM, a flat grid in metres. It finds the hull with the
monotone chain method, a standard way to find one. It gives the area of the hull on that flat grid.

A convex hull is **exact**. There is no tolerance to choose, no curve to approximate, and no answer to converge on.
So the two hulls either match corner for corner, or the two sides disagree about something real. That is why this
was the first check written among the widgets that owed one.

## How it is checked

We project the points ourselves with **pyproj**, the Python link to the PROJ projection library. We find the hull with
**shapely**, Python's link to GEOS, the geometry engine much GIS software uses. So the check tests the projection and
the hull together. It does not take Fieldwork's projected metres on trust. Nothing here uses Fieldwork's code.

The area compared is the area on the flat UTM grid, because that is what Fieldwork claims. It is not an area on the
curved Earth.

The data are the 250 cholera deaths in Soho, 1854, from the John Snow worked example.

## What should happen

1. **The same number of corners.** Fieldwork drops points that lie exactly on an edge of the hull, and so does
   shapely. A difference would be reported, not allowed for.
2. **Every corner within 1 mm** of one in our hull, and the other way round.
3. **The area within 0.05 m².**
4. **The span within 0.1 m.**
5. **The same number of located points.**

These criteria were fixed before the check was run.

In [ ]:
# pyproj, shapely, rasterio and friends ship in the Pyodide distribution, not as PyPI
# wheels on this site, so they load with pyodide_js.loadPackage. piplite.install would
# look in the site's local wheel index and fail. Outside Pyodide this cell does nothing.
#
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_10_hull import run

report = run(Path.cwd() / "fixtures" / "hull-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["points"], "located points")
print(f"vertices        fieldwork {report['fieldwork']['vertexCount']}   shapely {report['reference']['vertexCount']}")
print(f"area m2         fieldwork {report['fieldwork']['areaSquareMetres']:.2f}   shapely {report['reference']['areaSquareMetres']:.2f}")
print(f"span m          fieldwork {report['fieldwork']['spanMetres']:.1f}   shapely {report['reference']['spanMetres']:.1f}")
print()
print(f"worst vertex separation {report['worstVertexSeparationM']:.3e} m")
print(f"area difference         {report['areaDifferenceSquareMetres']:.3e} m2")
print(f"span difference         {report['spanDifferenceM']:.3e} m")

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.4.0:
**Agrees**.

Over 250 located points, both hulls have 11 corners. The corners are at most about 2 × 10⁻⁹ m apart, far below a
millimetre. The areas differ by 0.004 m², and the spans by 0.018 m.

Running the cell above repeats the check on the same data.

## How much of the hull is empty

A convex hull is a poor summary of a pattern that is not convex, such as cases along a street or deaths clustered
around one pump. The hull **fills it in**, and claims area where nothing was observed. Fieldwork says so in the
widget's own notes.

The cell below measures this for these deaths. It compares the hull's area with the area within 25 m of a death.
That 25 m is the same width the Reachable corridor widget uses for the final walk off a street.

In [ ]:
# How much of the hull is empty? Compare the hull's area with the area actually occupied,
# approximated as a 25 m buffer around the observed points -- the same half-width the
# corridor widget uses, and the same kind of claim.
import pyproj
from shapely.geometry import MultiPoint

fixture = json.loads((Path.cwd() / "fixtures" / "hull-soho-001.json").read_text())
forward = pyproj.Transformer.from_crs("OGC:CRS84", fixture["declaredParameters"]["metricCRS"], always_xy=True)
projected = [forward.transform(*p["coordinates"]) for p in fixture["input"]["points"] if p["coordinates"]]

hull = MultiPoint(projected).convex_hull
occupied = MultiPoint(projected).buffer(25)
print(f"hull             {hull.area:10.0f} m2")
print(f"within 25 m of a death  {occupied.area:10.0f} m2   ({occupied.area / hull.area:.0%} of the hull)")
print()
print(report["notEstablished"])

### What that number actually said

**91% of the hull lies within 25 m of a death.** So for these points, the objection to convex shapes is much weaker
than the general warning suggests. The deaths nearly fill their own hull.

Two things are true at once:

- The general warning stands. A hull is convex, and it will fill in a pattern that is not.
- It is **not** what happens here.

The 25 m is itself a claim. It was chosen because this project already argues about that figure, not because it is
right.

Had the share come out lower, the warning would have read better and been no more true. The check reports what it
measured.

## What this does not show

- That a convex hull is a sensible summary of these deaths. It fills in ground the outbreak did not cross, such as the
  river and the squares. Both sides claim that area in the same way. A check of arithmetic cannot confirm or
  contradict a claim about meaning.
- That the points are the right points.
- That the area means what a reader will assume it means. It is an area on a flat grid, not on the curved Earth.